# Variant 10 — Clean 45k + LoRA + Voting + Rule Verifier

Mục tiêu của notebook này:

- Fine-tune `NlpHUST/gpt2-vietnamese` trên Kaggle, Internet OFF, GPU, tổng runtime mục tiêu `<= 3h`.
- Không dùng external LLM/API/data ngoài.
- Không đổi backbone model.
- Chọn khoảng 40–50k mẫu sạch thay vì train toàn bộ data nhiễu.
- Target ngắn, answer-focused để giảm lỗi format.
- Inference có rule solver nhẹ + generate nhiều candidate + majority/rule verifier.
- Sinh `valid_output.json`, `valid_report.json`, và nếu có `test.json` thì sinh `test_predictions.json`.

Tên variant: `v10_clean45k_lora_vote_rule`.

In [ ]:
# =========================
# 0. CONFIG
# =========================
import os
import re
import gc
import json
import math
import time
import random
import zipfile
import inspect
import warnings
from pathlib import Path
from collections import Counter, defaultdict
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

VARIANT_NAME = "v10_clean45k_lora_vote_rule"
SAFE_EOS_ID = 50256

# Runtime knobs. Nếu Kaggle gần quá 3h, giảm TARGET_TRAIN_N hoặc NUM_CANDIDATES.
TARGET_TRAIN_N = 45_000
MAX_PER_ORIGINAL = 4
INTERNAL_DEV_GROUP_RATIO = 0.08
MAX_INTERNAL_DEV_EVAL = 800

MAX_LENGTH = 384
MAX_NEW_TOKENS = 40
NUM_TRAIN_EPOCHS = 1.20
PER_DEVICE_TRAIN_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01

# Inference knobs
NUM_CANDIDATES = 3
GEN_BATCH_SIZE = 16
USE_RULE_SOLVER = True

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Variant:", VARIANT_NAME)
print("Output dir:", OUTPUT_DIR)

In [ ]:
# =========================
# 1. FIND DATA + MODEL PATHS
# =========================
def find_first_existing(paths: List[str]) -> Optional[Path]:
    for p in paths:
        pp = Path(p)
        if pp.exists():
            return pp
    return None


def find_file_under(root: str, filename: str) -> Optional[Path]:
    rootp = Path(root)
    if not rootp.exists():
        return None
    matches = list(rootp.rglob(filename))
    if not matches:
        return None
    # Prefer shorter path / likely official dataset path.
    matches = sorted(matches, key=lambda p: (len(str(p)), str(p)))
    return matches[0]


def load_json_from_zip(zip_path: Path, member: str) -> List[Dict[str, Any]]:
    with zipfile.ZipFile(zip_path) as z:
        with z.open(member) as f:
            return json.load(f)


def load_train_valid():
    data_dir = Path("/kaggle/input/datasets/kimanh2002/dataset-math")

    def load(name):
        with open(data_dir / name, "r", encoding="utf-8") as f:
            return json.load(f)

    return load("train.json"), load("valid.json")


def find_model_dir() -> str:
    candidates = [
        "/kaggle/input/nlphustgpt2-vietnamese",
        "/kaggle/input/nlphust-gpt2-vietnamese",
        "/kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese",
        "/kaggle/input/dataset-math/nlphustgpt2-vietnamese",
        "/mnt/data/nlphustgpt2-vietnamese",
        "./nlphustgpt2-vietnamese",
    ]
    for c in candidates:
        p = Path(c)
        if (p / "config.json").exists():
            return str(p)

    # Search config.json under /kaggle/input and prefer dirs that look like gpt2-vietnamese.
    root = Path("/kaggle/input")
    if root.exists():
        configs = list(root.rglob("config.json"))
        scored = []
        for cfg in configs:
            d = cfg.parent
            s = str(d).lower()
            score = 0
            if "gpt2" in s: score += 2
            if "vietnam" in s or "nlphust" in s: score += 2
            scored.append((score, len(str(d)), d))
        if scored:
            scored.sort(key=lambda x: (-x[0], x[1], str(x[2])))
            print("Auto-detected model dir:", scored[0][2])
            return str(scored[0][2])

    # Last fallback: only works if internet/cache is available. Official run should not rely on this.
    return "NlpHUST/gpt2-vietnamese"

train_raw, valid_raw = load_train_valid()
MODEL_DIR = find_model_dir()

print("Train records:", len(train_raw))
print("Valid records:", len(valid_raw))
print("Model dir:", MODEL_DIR)
print("Train type counts:", Counter(x.get("type") for x in train_raw))
print("Valid type counts:", Counter(x.get("type") for x in valid_raw))

In [ ]:
# =========================
# 2. ANSWER EXTRACTION + SCORING
# =========================
ANSWER_ANCHORS = [
    r"Đáp\s*án\s*(?:là)?\s*[:：]?",
    r"Câu\s*trả\s*lời\s*(?:là)?\s*[:：]?",
    r"The\s*answer\s*is\s*[:：]?",
    r"Answer\s*[:：]?",
    r"####",
]

BAD_ARTIFACT_PATTERNS = [
    "ble x", "đóng hộp", "\\ đóng", "\u200b", "�", "NaN", "None",
]


def cleanup_text(s: Any) -> str:
    if s is None:
        return ""
    s = str(s)
    s = s.replace("\r", " ").replace("\t", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s


def strip_latex_noise(s: str) -> str:
    s = cleanup_text(s)
    s = s.replace("$", "")
    s = s.replace("\\left", "").replace("\\right", "")
    s = s.replace("\\,", "")
    s = s.replace("\\!", "")
    s = s.replace("\\boxed", "boxed")
    s = s.replace("\u2212", "-")
    return s.strip()


def normalize_answer_str(ans: str) -> str:
    ans = strip_latex_noise(ans)
    ans = ans.strip(" .,:;!?)】]}。")
    ans = ans.strip("({[【")
    ans = ans.replace("，", ",")
    ans = ans.replace("％", "%")
    ans = ans.replace(" ", "")
    # Vietnamese decimal comma: only convert if it looks like 4,5 not 1,000.
    if re.fullmatch(r"[-+]?\d+,\d+", ans):
        ans = ans.replace(",", ".")
    else:
        ans = ans.replace(",", "")
    return ans


def extract_boxed(text: str) -> Optional[str]:
    text = str(text)
    # Handles \boxed{...} and boxed{...}; intentionally simple.
    m = re.findall(r"(?:\\boxed|boxed)\s*\{([^{}]{1,80})\}", text)
    if m:
        return m[-1]
    return None


def extract_final_answer(text: Any) -> Optional[str]:
    text = cleanup_text(text)
    if not text:
        return None

    # Prefer answer after known anchors, taking the last anchor.
    last_pos = -1
    last_pat = None
    for pat in ANSWER_ANCHORS:
        for m in re.finditer(pat, text, flags=re.IGNORECASE):
            if m.end() > last_pos:
                last_pos = m.end()
                last_pat = pat

    if last_pos >= 0:
        tail = text[last_pos:last_pos + 160]
        boxed = extract_boxed(tail)
        if boxed:
            return normalize_answer_str(boxed)

        frac = re.search(r"-?\\frac\s*\{\s*-?\d+\s*\}\s*\{\s*-?\d+\s*\}", tail)
        if frac:
            return normalize_answer_str(frac.group(0))

        # tuple answer like (91,60) — keep compact string; numeric scorer may not support but output remains extractable.
        tup = re.search(r"\((-?\d+(?:\.\d+)?(?:,\s*-?\d+(?:\.\d+)?)+)\)", tail)
        if tup:
            return normalize_answer_str("(" + tup.group(1) + ")")

        # pi expression e.g. 36\pi, -\frac{\pi}{2}
        pi_expr = re.search(r"-?(?:\d+(?:\.\d+)?)?\\?pi(?:/\d+)?|-?\\frac\s*\{\s*\\?pi\s*\}\s*\{\s*\d+\s*\}", tail, flags=re.I)
        if pi_expr:
            return normalize_answer_str(pi_expr.group(0))

        num = re.search(r"[-+]?\d+(?:[\.,]\d+)?(?:/[-+]?\d+(?:[\.,]\d+)?)?%?", tail)
        if num:
            return normalize_answer_str(num.group(0))

    # Fallback: last boxed or last number in whole output.
    boxed = extract_boxed(text)
    if boxed:
        return normalize_answer_str(boxed)

    nums = re.findall(r"[-+]?\d+(?:[\.,]\d+)?(?:/[-+]?\d+(?:[\.,]\d+)?)?%?", text)
    if nums:
        return normalize_answer_str(nums[-1])
    return None


def answer_to_float(ans: Any) -> Optional[float]:
    if ans is None:
        return None
    s = normalize_answer_str(str(ans))
    if not s:
        return None
    if s.startswith("(") and s.endswith(")"):
        return None
    s = s.replace("\\pi", "pi").replace("π", "pi")
    s = s.replace("\\dfrac", "\\frac")
    s = s.replace("^", "**")

    # \frac{a}{b}
    def repl_frac(m):
        return f"(({m.group(1)})/({m.group(2)}))"
    s = re.sub(r"\\frac\{([^{}]+)\}\{([^{}]+)\}", repl_frac, s)
    s = re.sub(r"frac\{([^{}]+)\}\{([^{}]+)\}", repl_frac, s)

    if s.endswith("%"):
        try:
            return float(s[:-1]) / 100.0
        except Exception:
            return None

    # 3/4 style fraction.
    if re.fullmatch(r"[-+]?\d+(?:\.\d+)?/[-+]?\d+(?:\.\d+)?", s):
        a, b = s.split("/")
        try:
            return float(a) / float(b)
        except Exception:
            return None

    # simple pi coefficients: 36pi, -pi/2
    if "pi" in s:
        try:
            expr = s.replace("pi", "*math.pi")
            expr = re.sub(r"(^|[+\-*/(])\*math\.pi", r"\1math.pi", expr)
            expr = re.sub(r"(\d)math\.pi", r"\1*math.pi", expr)
            if re.fullmatch(r"[0-9\.\+\-\*/\(\)mathpi ]+", expr):
                return float(eval(expr, {"__builtins__": {}}, {"math": math}))
        except Exception:
            return None

    try:
        return float(s)
    except Exception:
        return None


def score_one(pred_answer: Any, gold_answer: Any) -> Tuple[int, Optional[float]]:
    p = answer_to_float(pred_answer)
    g = answer_to_float(gold_answer)
    if p is None or g is None or not np.isfinite(p) or not np.isfinite(g):
        return 0, None
    rel = abs(p - g) / max(1.0, abs(g))
    if rel <= 0.01:
        return 10, rel
    if rel <= 0.10:
        return 5, rel
    if rel <= 0.50:
        return 1, rel
    return 0, rel

# Quick sanity checks.
for s in ["#### 37 Đáp án là: 37", "Câu trả lời là: \\frac{9}{20}", "Đáp án là: 36\\pi", "Đáp án là: -2"]:
    a = extract_final_answer(s)
    print(s, "=>", a, answer_to_float(a))

In [ ]:
# =========================
# 3. CLEAN DATA + SELECT 45K TRAINING SUBSET
# =========================
def build_df(records: List[Dict[str, Any]], name: str) -> pd.DataFrame:
    rows = []
    for i, r in enumerate(records):
        query = cleanup_text(r.get("query_vi", ""))
        resp = cleanup_text(r.get("response_vi", ""))
        ans = extract_final_answer(resp)
        rows.append({
            "idx": i,
            "query_vi": query,
            "response_vi": resp,
            "type": r.get("type", "UNKNOWN"),
            "original_question_en": cleanup_text(r.get("original_question_en", "")),
            "original_question_vi": cleanup_text(r.get("original_question_vi", "")),
            "query_en": cleanup_text(r.get("query_en", "")),
            "gold_answer": ans,
            "query_len": len(query),
            "response_len": len(resp),
        })
    df = pd.DataFrame(rows)
    print(f"{name}: {len(df)} rows")
    print(f"{name}: extract fail =", df["gold_answer"].isna().sum())
    print(df["type"].value_counts())
    return df


def add_quality_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["group_key"] = df["original_question_en"].where(df["original_question_en"].str.len() > 0, df["query_en"])
    df["group_key"] = df["group_key"].where(df["group_key"].str.len() > 0, df["query_vi"])
    df["has_bad_artifact"] = df["response_vi"].apply(lambda s: any(pat in str(s) for pat in BAD_ARTIFACT_PATTERNS))
    df["has_answer_anchor"] = df["response_vi"].str.contains(r"Đáp\s*án|Câu\s*trả\s*lời|####|The answer", case=False, regex=True, na=False)
    df["answer_float"] = df["gold_answer"].apply(answer_to_float)
    df["answer_is_numeric"] = df["answer_float"].apply(lambda x: x is not None and np.isfinite(x))

    # Lower is better. Prefer clean, short-ish, anchored answers.
    df["quality_score"] = 0.0
    df.loc[df["has_bad_artifact"], "quality_score"] += 100
    df.loc[~df["has_answer_anchor"], "quality_score"] += 30
    df.loc[~df["answer_is_numeric"], "quality_score"] += 20
    df["quality_score"] += np.maximum(0, df["response_len"] - 1200) / 100.0
    df["quality_score"] += np.maximum(0, df["query_len"] - 900) / 100.0
    return df

train_df = add_quality_columns(build_df(train_raw, "train"))
valid_df = add_quality_columns(build_df(valid_raw, "valid"))

# Strict clean for training. Keep valid as-is for evaluation/reporting.
clean_train = train_df[
    train_df["gold_answer"].notna()
    & train_df["query_vi"].str.len().between(5, 1200)
    & train_df["response_vi"].str.len().between(3, 2200)
    & (~train_df["has_bad_artifact"])
].copy()

print("Clean train:", len(clean_train), "/", len(train_df))
print("Clean train type counts:")
print(clean_train["type"].value_counts())
print("Unique original groups:", clean_train["group_key"].nunique())

In [ ]:
# =========================
# 4. GROUP SPLIT + TYPE-BALANCED SUBSET
# =========================
def group_train_dev_split(df: pd.DataFrame, dev_ratio: float = 0.08, seed: int = 42) -> Tuple[pd.DataFrame, pd.DataFrame]:
    groups = df["group_key"].dropna().unique().tolist()
    rng = random.Random(seed)
    rng.shuffle(groups)
    n_dev = max(1, int(len(groups) * dev_ratio))
    dev_groups = set(groups[:n_dev])
    dev = df[df["group_key"].isin(dev_groups)].copy()
    train = df[~df["group_key"].isin(dev_groups)].copy()
    return train, dev


def compute_type_quotas(valid_df: pd.DataFrame, target_n: int) -> Dict[str, int]:
    vc = valid_df["type"].value_counts()
    total = vc.sum()
    quotas = {t: int(round(target_n * c / total)) for t, c in vc.items()}
    diff = target_n - sum(quotas.values())
    if diff != 0:
        top = vc.index[0]
        quotas[top] = quotas.get(top, 0) + diff
    return quotas


def select_subset(df: pd.DataFrame, quotas: Dict[str, int], max_per_original: int = 4, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    df = df.copy()
    df["rand"] = rng.random(len(df))
    selected_idxs = []
    group_counts = defaultdict(int)

    # First pass: satisfy type quotas.
    for t, quota in quotas.items():
        part = df[df["type"] == t].sort_values(["quality_score", "rand", "response_len"])
        chosen = 0
        for row in part.itertuples():
            if chosen >= quota:
                break
            if group_counts[row.group_key] >= max_per_original:
                continue
            selected_idxs.append(row.Index)
            group_counts[row.group_key] += 1
            chosen += 1
        print(f"type={t:15s} quota={quota:5d} chosen={chosen:5d}")

    # Second pass: fill any shortage from globally best remaining rows.
    target_n = sum(quotas.values())
    already = set(selected_idxs)
    if len(selected_idxs) < target_n:
        remain = df[~df.index.isin(already)].sort_values(["quality_score", "rand", "response_len"])
        for row in remain.itertuples():
            if len(selected_idxs) >= target_n:
                break
            if group_counts[row.group_key] >= max_per_original:
                continue
            selected_idxs.append(row.Index)
            group_counts[row.group_key] += 1

    subset = df.loc[selected_idxs].sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return subset

train_pool, internal_dev = group_train_dev_split(clean_train, INTERNAL_DEV_GROUP_RATIO, SEED)
quotas = compute_type_quotas(valid_df, TARGET_TRAIN_N)
train_subset = select_subset(train_pool, quotas, MAX_PER_ORIGINAL, SEED)

# Small internal dev for quick evaluation/loss only.
internal_dev_eval = internal_dev.sort_values(["quality_score", "response_len"]).groupby("type", group_keys=False).head(150)
if len(internal_dev_eval) > MAX_INTERNAL_DEV_EVAL:
    internal_dev_eval = internal_dev_eval.sample(MAX_INTERNAL_DEV_EVAL, random_state=SEED).reset_index(drop=True)
else:
    internal_dev_eval = internal_dev_eval.reset_index(drop=True)

print("\nFinal train subset:", len(train_subset))
print(train_subset["type"].value_counts())
print("Internal dev:", len(internal_dev), "eval subset:", len(internal_dev_eval))
print("Overlap check:", len(set(train_subset.group_key) & set(internal_dev.group_key)))

# Save audit files.
train_subset[["query_vi", "type", "gold_answer", "group_key", "quality_score"]].to_csv(OUTPUT_DIR / "v10_train_subset_audit.csv", index=False)
internal_dev_eval[["query_vi", "type", "gold_answer", "group_key", "quality_score"]].to_csv(OUTPUT_DIR / "v10_internal_dev_audit.csv", index=False)

In [ ]:
# =========================
# 5. PROMPT / TARGET FORMAT
# =========================
def make_prompt(query_vi: str) -> str:
    query_vi = cleanup_text(query_vi)
    return (
        "Nhiệm vụ: Giải bài toán tiếng Việt và trả về đáp án cuối.\n"
        f"Bài toán: {query_vi}\n"
        "Lời giải:"
    )


def make_target(answer: str) -> str:
    answer = normalize_answer_str(str(answer))
    return f" Đáp án là: {answer}"


def make_model_output_from_answer(answer: str, prefix: str = "Tính theo dữ kiện trong đề.") -> str:
    answer = normalize_answer_str(str(answer))
    return f"Lời giải: {prefix}\nĐáp án là: {answer}"

print(make_prompt("Giải bài toán 1 + 1 bằng bao nhiêu?"))
print(make_target("2"))

In [ ]:
# =========================
# 6. LOAD TOKENIZER + MODEL + LORA
# =========================
import torch
from torch.utils.data import Dataset

from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments

try:
    from peft import LoraConfig, get_peft_model, TaskType
except Exception as e:
    raise RuntimeError(
        "Không import được peft. Trên Kaggle thường đã có peft; nếu không, hãy attach một Kaggle dataset chứa wheel/package peft offline."
    ) from e

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Load tokenizer/model. Official Kaggle run should load from local dataset path.
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=Path(MODEL_DIR).exists())
if tokenizer.eos_token_id is None:
    tokenizer.eos_token_id = SAFE_EOS_ID
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token if tokenizer.eos_token is not None else "<|endoftext|>"
# Enforce safe ids used by the competition note.
tokenizer.pad_token_id = SAFE_EOS_ID
tokenizer.eos_token_id = SAFE_EOS_ID
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    local_files_only=Path(MODEL_DIR).exists(),
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)
model.config.pad_token_id = SAFE_EOS_ID
model.config.eos_token_id = SAFE_EOS_ID
if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()
model.config.use_cache = False

# Detect LoRA target modules for GPT-2 style model.
module_names = [name for name, _ in model.named_modules()]
target_modules = []
for cand in ["c_attn", "c_proj"]:
    if any(name.endswith(cand) for name in module_names):
        target_modules.append(cand)
if not target_modules:
    target_modules = ["c_attn"]
print("LoRA target modules:", target_modules)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# =========================
# 7. DATASET WITH TARGET-ONLY LOSS
# =========================
class MathAnswerOnlyDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_length: int = 384):
        self.records = df[["query_vi", "gold_answer", "type"]].to_dict("records")
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        prompt = make_prompt(r["query_vi"])
        target = make_target(r["gold_answer"])
        target_ids = self.tokenizer(target, add_special_tokens=False)["input_ids"] + [SAFE_EOS_ID]
        max_prompt_len = max(8, self.max_length - len(target_ids))
        prompt_ids = self.tokenizer(
            prompt,
            add_special_tokens=False,
            truncation=True,
            max_length=max_prompt_len,
        )["input_ids"]
        input_ids = prompt_ids + target_ids
        labels = [-100] * len(prompt_ids) + target_ids
        if len(input_ids) > self.max_length:
            input_ids = input_ids[-self.max_length:]
            labels = labels[-self.max_length:]
        attention_mask = [1] * len(input_ids)
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def collate_batch(features: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
    max_len = max(len(f["input_ids"]) for f in features)
    input_ids, attention_mask, labels = [], [], []
    for f in features:
        n = len(f["input_ids"])
        pad = max_len - n
        input_ids.append(torch.cat([f["input_ids"], torch.full((pad,), SAFE_EOS_ID, dtype=torch.long)]))
        attention_mask.append(torch.cat([f["attention_mask"], torch.zeros(pad, dtype=torch.long)]))
        labels.append(torch.cat([f["labels"], torch.full((pad,), -100, dtype=torch.long)]))
    return {
        "input_ids": torch.stack(input_ids),
        "attention_mask": torch.stack(attention_mask),
        "labels": torch.stack(labels),
    }

train_dataset = MathAnswerOnlyDataset(train_subset, tokenizer, MAX_LENGTH)
eval_dataset = MathAnswerOnlyDataset(internal_dev_eval, tokenizer, MAX_LENGTH)

sample = train_dataset[0]
print("Sample input len:", len(sample["input_ids"]))
print(tokenizer.decode(sample["input_ids"][:200]))
print("Label tokens supervised:", int((sample["labels"] != -100).sum()))

In [ ]:
# =========================
# 8. TRAIN LORA
# =========================
# Compatibility with both older and newer transformers versions.
args_kwargs = dict(
    output_dir=str(OUTPUT_DIR / VARIANT_NAME),
    overwrite_output_dir=True,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    logging_steps=50,
    eval_steps=300,
    save_steps=10_000,
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
    report_to="none",
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

sig = inspect.signature(TrainingArguments.__init__)
valid_params = set(sig.parameters.keys())

if "eval_strategy" in valid_params:
    args_kwargs["eval_strategy"] = "steps"
elif "evaluation_strategy" in valid_params:
    args_kwargs["evaluation_strategy"] = "steps"

if "save_strategy" in valid_params:
    args_kwargs["save_strategy"] = "no"

unsupported = sorted([k for k in args_kwargs if k not in valid_params])
if unsupported:
    print("Dropping unsupported TrainingArguments kwargs:", unsupported)

args_kwargs = {k: v for k, v in args_kwargs.items() if k in valid_params}

training_args = TrainingArguments(**args_kwargs)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collate_batch,
)

start = time.time()
train_result = trainer.train()
train_minutes = (time.time() - start) / 60
print("Training minutes:", train_minutes)
print(train_result)

# Save LoRA adapter + tokenizer.
adapter_dir = OUTPUT_DIR / f"{VARIANT_NAME}_adapter"
adapter_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print("Saved adapter to:", adapter_dir)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# =========================
# 9. CONSERVATIVE RULE SOLVER
# =========================
def safe_eval_arithmetic(expr: str) -> Optional[float]:
    expr = expr.strip()
    expr = expr.replace("×", "*").replace("x", "*").replace("X", "*").replace("÷", "/")
    expr = expr.replace("^", "**")
    # Only allow numbers and arithmetic signs.
    if not re.fullmatch(r"[0-9\.\+\-\*/\(\)\s]+", expr):
        return None
    try:
        val = eval(expr, {"__builtins__": {}}, {})
        if isinstance(val, (int, float)) and np.isfinite(val):
            return float(val)
    except Exception:
        return None
    return None


def format_number(x: float) -> str:
    if x is None or not np.isfinite(x):
        return ""
    if abs(x - round(x)) < 1e-9:
        return str(int(round(x)))
    return (f"{x:.10f}").rstrip("0").rstrip(".")


def rule_solver(query: str, typ: str = "") -> Optional[str]:
    q = cleanup_text(query)
    q_low = q.lower()

    # Direct expression questions: "1 + 1 bằng bao nhiêu", "Tính 12*3".
    expr_candidates = []
    for m in re.finditer(r"(?<!\d)(\d+(?:\.\d+)?\s*[\+\-\*/×÷]\s*\d+(?:\.\d+)?(?:\s*[\+\-\*/×÷]\s*\d+(?:\.\d+)?)*)", q):
        expr_candidates.append(m.group(1))
    if expr_candidates and any(w in q_low for w in ["tính", "bằng bao nhiêu", "giá trị", "kết quả"]):
        val = safe_eval_arithmetic(expr_candidates[-1])
        if val is not None:
            return make_model_output_from_answer(format_number(val), "Tính trực tiếp biểu thức trong đề.")

    # LCM/GCD style.
    nums = [int(x) for x in re.findall(r"(?<![\d.])-?\d+(?![\d.])", q)]
    pos_nums = [abs(x) for x in nums if abs(x) > 0]
    if len(pos_nums) >= 2 and any(w in q_low for w in ["bội số chung nhỏ nhất", "bội chung nhỏ nhất", "lcm"]):
        val = pos_nums[0]
        for n in pos_nums[1:]:
            val = math.lcm(val, n)
        return make_model_output_from_answer(str(val), "Áp dụng quy tắc bội chung nhỏ nhất.")
    if len(pos_nums) >= 2 and any(w in q_low for w in ["ước chung lớn nhất", "uoc chung lon nhat", "gcd"]):
        val = pos_nums[0]
        for n in pos_nums[1:]:
            val = math.gcd(val, n)
        return make_model_output_from_answer(str(val), "Áp dụng quy tắc ước chung lớn nhất.")

    # Very specific board-game remaining distance pattern.
    if len(pos_nums) >= 2 and any(w in q_low for w in ["cần đi thêm", "cần di chuyển thêm", "bao nhiêu ô"]):
        # If text says total spaces and current moved, last two relevant numbers are often total/current in simplified examples.
        if len(pos_nums) == 2 and pos_nums[0] > pos_nums[1]:
            return make_model_output_from_answer(str(pos_nums[0] - pos_nums[1]), "Lấy tổng số ô trừ số ô đã đi.")

    # Isosceles triangle simplified pattern: two equal sides a and remaining b.
    if "chu vi" in q_low and "tam giác" in q_low and len(pos_nums) == 2 and any(w in q_low for w in ["hai cạnh bằng", "2 cạnh bằng"]):
        a, b = pos_nums
        return make_model_output_from_answer(str(2 * a + b), "Chu vi bằng tổng ba cạnh.")

    return None

# Sanity checks.
for q in [
    "Giải bài toán 1 + 1 bằng bao nhiêu?",
    "Tìm bội số chung nhỏ nhất của 24 và 90.",
    "Một tam giác có hai cạnh bằng 7 và cạnh còn lại bằng 5. Chu vi tam giác là bao nhiêu?",
]:
    print(q, "=>", rule_solver(q))

In [ ]:
# =========================
# 10. GENERATION + VOTING VERIFIER
# =========================
# For generation, left padding is more stable with GPT-style causal LM.
tokenizer.padding_side = "left"
model.eval()
if hasattr(model.config, "use_cache"):
    model.config.use_cache = True


def build_generation_prompts(records: List[Dict[str, Any]]) -> List[str]:
    return [make_prompt(r.get("query_vi", "")) for r in records]


def generate_one_pass(prompts: List[str], do_sample: bool, temperature: float = 0.7, top_p: float = 0.90) -> List[str]:
    outs = []
    device = model.device
    for i in range(0, len(prompts), GEN_BATCH_SIZE):
        batch_prompts = prompts[i:i + GEN_BATCH_SIZE]
        enc = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH - MAX_NEW_TOKENS,
            add_special_tokens=False,
        ).to(device)
        gen_kwargs = dict(
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=do_sample,
            num_beams=1,
            pad_token_id=SAFE_EOS_ID,
            eos_token_id=SAFE_EOS_ID,
        )
        if do_sample:
            gen_kwargs.update(dict(temperature=temperature, top_p=top_p))
        with torch.no_grad():
            gen = model.generate(**enc, **gen_kwargs)
        for j in range(len(batch_prompts)):
            input_len = int(enc["attention_mask"][j].sum().item())
            # With left padding, generated row length includes pads. Decode only final new tokens approximately.
            tail_ids = gen[j][-MAX_NEW_TOKENS:]
            tail = tokenizer.decode(tail_ids, skip_special_tokens=True)
            # Remove prompt leakage if present.
            if "Lời giải:" in tail:
                tail = tail.split("Lời giải:")[-1]
            outs.append("Lời giải:" + tail.strip())
    return outs


def candidate_quality(output: str) -> float:
    ans = extract_final_answer(output)
    score = 0.0
    if ans is not None:
        score += 50
    if "Đáp án" in output:
        score += 20
    if "Lời giải" in output:
        score += 5
    # Penalize very noisy outputs.
    score -= max(0, len(output) - 220) / 20
    if any(pat in output for pat in BAD_ARTIFACT_PATTERNS):
        score -= 100
    return score


def choose_by_voting(candidates: List[str]) -> str:
    parsed = []
    for out in candidates:
        ans = extract_final_answer(out)
        if ans is None:
            continue
        # Use numeric rounded key when possible to merge 2 and 2.0.
        f = answer_to_float(ans)
        key = f"num:{round(f, 8)}" if f is not None else f"str:{normalize_answer_str(ans)}"
        parsed.append((key, ans, out))

    if parsed:
        key_counts = Counter(k for k, _, _ in parsed)
        best_key, _ = key_counts.most_common(1)[0]
        same = [(ans, out) for k, ans, out in parsed if k == best_key]
        best_out = max([out for _, out in same], key=candidate_quality)
        best_ans = extract_final_answer(best_out)
        if best_ans is not None:
            return make_model_output_from_answer(best_ans, "Chọn đáp án ổn định nhất từ các candidate.")
        return best_out

    # If nothing extractable, return best-looking raw candidate.
    if candidates:
        return max(candidates, key=candidate_quality)
    return make_model_output_from_answer("0", "Không sinh được đáp án hợp lệ.")


def predict_records(records: List[Dict[str, Any]], use_rule_solver: bool = True) -> List[Dict[str, Any]]:
    predictions = [None] * len(records)
    need_model = []
    need_indices = []

    for i, r in enumerate(records):
        q = r.get("query_vi", "")
        typ = r.get("type", "")
        solved = rule_solver(q, typ) if use_rule_solver else None
        if solved is not None:
            predictions[i] = solved
        else:
            need_model.append(r)
            need_indices.append(i)

    print(f"Rule solved: {len(records) - len(need_model)} / {len(records)}")
    if need_model:
        prompts = build_generation_prompts(need_model)
        all_passes = []
        # 1 greedy + sampled passes.
        all_passes.append(generate_one_pass(prompts, do_sample=False))
        for k in range(NUM_CANDIDATES - 1):
            temp = 0.65 + 0.10 * k
            all_passes.append(generate_one_pass(prompts, do_sample=True, temperature=temp, top_p=0.90))

        for local_i, global_i in enumerate(need_indices):
            cands = [pass_outs[local_i] for pass_outs in all_passes]
            predictions[global_i] = choose_by_voting(cands)

    out_records = []
    for i, r in enumerate(records):
        out_records.append({
            "id": int(r.get("id", i)) if str(r.get("id", i)).isdigit() else i,
            "query_vi": r.get("query_vi", ""),
            "type": r.get("type", ""),
            "model_output": predictions[i],
        })
    return out_records

In [ ]:
# =========================
# 11. VALIDATION EVALUATION
# =========================
def evaluate_predictions(pred_records: List[Dict[str, Any]], gold_records: List[Dict[str, Any]]) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    rows = []
    for i, (pred, gold) in enumerate(zip(pred_records, gold_records)):
        gold_ans = extract_final_answer(gold.get("response_vi", ""))
        pred_ans = extract_final_answer(pred.get("model_output", ""))
        s, rel = score_one(pred_ans, gold_ans)
        rows.append({
            "id": pred.get("id", i),
            "type": gold.get("type", pred.get("type", "")),
            "query_vi": gold.get("query_vi", pred.get("query_vi", "")),
            "gold_answer": gold_ans,
            "pred_answer": pred_ans,
            "score": s,
            "relative_error": rel,
            "extract_ok": pred_ans is not None,
            "model_output": pred.get("model_output", ""),
        })
    df = pd.DataFrame(rows)
    report = {
        "variant": VARIANT_NAME,
        "n": int(len(df)),
        "raw_score": float(df["score"].sum()),
        "score_10": float(df["score"].sum() / max(1, len(df))),
        "extract_rate": float(df["extract_ok"].mean()),
        "score_counts": {str(k): int(v) for k, v in df["score"].value_counts().sort_index().items()},
        "by_type": {},
        "config": {
            "target_train_n": TARGET_TRAIN_N,
            "actual_train_n": int(len(train_subset)),
            "max_per_original": MAX_PER_ORIGINAL,
            "max_length": MAX_LENGTH,
            "max_new_tokens": MAX_NEW_TOKENS,
            "num_candidates": NUM_CANDIDATES,
            "epochs": NUM_TRAIN_EPOCHS,
            "lr": LEARNING_RATE,
            "batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
            "grad_accum": GRADIENT_ACCUMULATION_STEPS,
            "lora_r": 16,
            "lora_alpha": 32,
        }
    }
    for t, g in df.groupby("type"):
        report["by_type"][t] = {
            "n": int(len(g)),
            "score_10": float(g["score"].sum() / max(1, len(g))),
            "extract_rate": float(g["extract_ok"].mean()),
            "score_counts": {str(k): int(v) for k, v in g["score"].value_counts().sort_index().items()},
        }
    return df, report

# Prepare valid records with ids.
valid_records = []
for i, r in enumerate(valid_raw):
    rr = dict(r)
    rr["id"] = i
    valid_records.append(rr)

start = time.time()
valid_predictions = predict_records(valid_records, use_rule_solver=USE_RULE_SOLVER)
valid_minutes = (time.time() - start) / 60
print("Valid inference minutes:", valid_minutes)

valid_eval_df, valid_report = evaluate_predictions(valid_predictions, valid_raw)

valid_output_path = OUTPUT_DIR / "valid_output.json"
valid_report_path = OUTPUT_DIR / "valid_report.json"
valid_error_path = OUTPUT_DIR / "valid_error_analysis.csv"

with open(valid_output_path, "w", encoding="utf-8") as f:
    json.dump(valid_predictions, f, ensure_ascii=False, indent=2)
with open(valid_report_path, "w", encoding="utf-8") as f:
    json.dump(valid_report, f, ensure_ascii=False, indent=2)
valid_eval_df.sort_values(["score", "type", "relative_error"], na_position="first").to_csv(valid_error_path, index=False)

print(json.dumps(valid_report, ensure_ascii=False, indent=2)[:4000])
print("Saved:", valid_output_path)
print("Saved:", valid_report_path)
print("Saved:", valid_error_path)

In [ ]:
# =========================
# 12. SHOW WORST ERRORS FOR DEBUGGING
# =========================
pd.set_option("display.max_colwidth", 180)
cols = ["id", "type", "score", "gold_answer", "pred_answer", "relative_error", "query_vi", "model_output"]
display(valid_eval_df[valid_eval_df["score"] == 0][cols].head(20))

display(valid_eval_df.groupby("type").agg(
    n=("score", "size"),
    score_10=("score", lambda x: x.sum() / max(1, len(x))),
    extract_rate=("extract_ok", "mean"),
).sort_values("score_10"))

In [ ]:
# =========================
# 13. PREDICT TEST IF test.json EXISTS
# =========================
def find_test_records() -> Optional[List[Dict[str, Any]]]:
    test_path = find_file_under("/kaggle/input", "test.json")
    if test_path is None:
        test_path = find_first_existing(["/mnt/data/test.json", "./test.json"])
    if test_path is not None:
        print("Test path:", test_path)
        with open(test_path, "r", encoding="utf-8") as f:
            return json.load(f)

    zip_path = find_file_under("/kaggle/input", "data.zip")
    if zip_path is None:
        zip_path = find_first_existing(["/mnt/data/data.zip", "./data.zip"])
    if zip_path is not None:
        try:
            with zipfile.ZipFile(zip_path) as z:
                if "test.json" in z.namelist():
                    print("Test from zip:", zip_path)
                    return load_json_from_zip(zip_path, "test.json")
        except Exception:
            pass
    return None

test_raw = find_test_records()
if test_raw is None:
    print("Không tìm thấy test.json. Phase 1 chỉ sinh valid_output.json và valid_report.json.")
else:
    test_records = []
    for i, r in enumerate(test_raw):
        rr = dict(r)
        rr["id"] = int(r.get("id", i)) if str(r.get("id", i)).isdigit() else i
        test_records.append(rr)
    test_predictions = predict_records(test_records, use_rule_solver=USE_RULE_SOLVER)
    test_path = OUTPUT_DIR / "test_predictions.json"
    with open(test_path, "w", encoding="utf-8") as f:
        json.dump(test_predictions, f, ensure_ascii=False, indent=2)
    print("Saved test predictions:", test_path)
    print("Num test predictions:", len(test_predictions))
    print(test_predictions[:2])

In [ ]:
# =========================
# 14. FINAL RUNTIME SUMMARY
# =========================
summary = {
    "variant": VARIANT_NAME,
    "train_records_raw": len(train_raw),
    "valid_records_raw": len(valid_raw),
    "clean_train_records": int(len(clean_train)),
    "selected_train_records": int(len(train_subset)),
    "internal_dev_eval_records": int(len(internal_dev_eval)),
    "valid_score_10": valid_report.get("score_10"),
    "valid_extract_rate": valid_report.get("extract_rate"),
    "output_files": [
        str(OUTPUT_DIR / "valid_output.json"),
        str(OUTPUT_DIR / "valid_report.json"),
        str(OUTPUT_DIR / "valid_error_analysis.csv"),
        str(OUTPUT_DIR / "test_predictions.json"),
    ],
}
with open(OUTPUT_DIR / "v10_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print(json.dumps(summary, ensure_ascii=False, indent=2))